# MEA Alpha Machine — 一二三阶挖掘流水线(MEA / TOP400)

基于《WQ第五六节课代码/顾问参考代码》的 Alpha Machine 流程改造:

- **Region**: `MEA`, **Universe**: `TOP400`(可改 `TOP300`), **Delay**: 1(MEA 仅支持 delay=1)
- **Neutralization**: `SUBINDUSTRY`(MEA 支持 NONE/MARKET/SECTOR/INDUSTRY/SUBINDUSTRY/COUNTRY)
- 一阶:ts 算子 + basic 算子;二阶:group 算子(MEA 用 sta2_top400 统计行业分组,无 fnd28);三阶:trade_when
- 登录凭据:默认用 mea_machine_lib.py 中填写的账号,也可用环境变量 `BRAIN_EMAIL` / `BRAIN_PASSWORD` 覆盖

> 提示:MEA 一阶全量约 10~20 万个 alpha,跑完需要较长时间;可在下方 `FIRST_ORDER_LIMIT` 处截断先试跑。

In [ ]:
from mea_machine_lib import *
import datetime

# 模拟目标设置
REGION = "MEA"
UNIVERSE = "TOP400"   # MEA 可选 TOP400 / TOP300
NEUT = "SUBINDUSTRY"   # MEA 可选 NONE/MARKET/SECTOR/INDUSTRY/SUBINDUSTRY/COUNTRY
DELAY = 1

# 一阶表达式数量上限(None = 不截断;建议先设 30000 试跑)
FIRST_ORDER_LIMIT = None

# 用于 get_alphas 的日期窗口(模拟创建日期),默认最近 7 天
TODAY = datetime.date.today()
END_D = TODAY.strftime("%m-%d")
START_D = (TODAY - datetime.timedelta(days=7)).strftime("%m-%d")
print("date window:", START_D, "->", END_D)

In [ ]:
s = login()

## 1. 拉取 MEA 数据集

`alphaCount>1000` 的数据集(MEA 数据集普遍较小,原 USA 阈值 10000 只会有 pv1 一个达标)。

In [ ]:
def get_datasets(
    s,
    instrument_type: str = 'EQUITY',
    region: str = 'MEA',
    delay: int = 1,
    universe: str = 'TOP400'
):
    url = ("https://api.worldquantbrain.com/data-sets?"
           + f"instrumentType={instrument_type}&region={region}&delay={str(delay)}&universe={universe}")
    result = s.get(url)
    datasets_df = pd.DataFrame(result.json()['results'])
    return datasets_df

datasets_df = get_datasets(s)
print("MEA datasets total:", len(datasets_df))
datasets_df[['id','name','category','fieldCount','alphaCount','coverage']].sort_values('alphaCount', ascending=False).head(20)

In [ ]:
datasets = datasets_df[datasets_df['alphaCount']>1000]['id'].tolist()
print(len(datasets))
print(datasets)

In [ ]:
pc_fields = []
for dd in datasets:
    df = get_datafields(s, dataset_id = dd, region='MEA', universe='TOP400', delay=1)
    temp = process_datafields(df, "matrix") + process_datafields(df, "vector")
    pc_fields = pc_fields + temp
    print(dd, len(temp))

print("total pc_fields:", len(pc_fields))
print(pc_fields[0])

## 2. 一阶:ts 算子 + basic 算子

对每个字段套用 `basic_ops + ts_ops`(含 rank/zscore/ts_rank/ts_zscore 等)。

In [ ]:
first_order = get_first_order(pc_fields, ts_ops, region='mea')
print(len(first_order))
print(first_order[:10])

# 可选:截断一阶数量(打散后取前 N,保证覆盖面)
if FIRST_ORDER_LIMIT:
    random.shuffle(first_order)
    first_order = first_order[:FIRST_ORDER_LIMIT]
    print("after limit:", len(first_order))

In [ ]:
# Pad initial decay with alpha
init_decay = 4
fo_alpha_list = []
for alpha in first_order:
    fo_alpha_list.append((alpha, init_decay))
print(len(fo_alpha_list))
print(fo_alpha_list[:5])

In [ ]:
# Load alphas to task pools
pools = load_task_pool(fo_alpha_list, 10, 9)
print(len(pools))
print(pools[0])

## 3. 模拟一阶(MEA / TOP400 / SUBINDUSTRY)

> 这一步会提交大量模拟,耗时较长。

In [ ]:
multi_simulate(pools, NEUT, REGION, UNIVERSE, 0)

In [ ]:
## get promising alphas to improve in the next order
fo_tracker = get_alphas(START_D, END_D, 1.2, 0.5, REGION, 1000, "track")
print(len(fo_tracker['next']))
print(len(fo_tracker['decay']))

## 4. 二阶:group 算子

MEA 分组:market/sector/industry/subindustry + cap/sector_cap/vol bucket + `sta2_top400_*` 统计行业聚类(pv30)。`prune` 按 MEA 数据集字段前缀逐组去重(mdl25/fnd6/fnd72/analyst/est/pv96/ern3 等)。

In [ ]:
# MEA 数据集字段前缀(用于 prune 去重)
MEA_PREFIXES = ['mdl25', 'mdl31', 'fnd6', 'fnd72', 'analyst', 'est_', 'pv96', 'ern3']

def mea_prune(recs, prefix_list, keep_num):
    # 对每个前缀分别 prune,再合并(与原 prune 单前缀逻辑等价)
    merged = []
    for prefix in prefix_list:
        layer = prune(recs, REGION, prefix, keep_num)
        for expr, decay in layer.get(REGION, []):
            merged.append([expr, decay])
    return merged

fo_layer = mea_prune(fo_tracker['next'] + fo_tracker['decay'], MEA_PREFIXES, 5)
print(len(fo_layer))

In [ ]:
so_alpha_list = []
group_ops = ["group_neutralize", "group_rank", "group_normalize", "group_scale", "group_zscore"]
for expr, decay in fo_layer:
    for alpha in get_group_second_order_factory([expr], group_ops, 'mea'):
        so_alpha_list.append((alpha, decay))

print(len(so_alpha_list))
print(so_alpha_list[:3])

In [ ]:
so_pools = load_task_pool(so_alpha_list, 9, 10)
multi_simulate(so_pools, NEUT, REGION, UNIVERSE, 0)

In [ ]:
## get promising alphas from second order to improve in the third order
so_tracker = get_alphas(START_D, END_D, 1.4, 0.7, REGION, 360, "track")
print(len(so_tracker['next']))
print(len(so_tracker['decay']))

## 5. 三阶:trade_when

MEA 无专属 sentiment 事件集,使用通用 open/exit 事件(含 `ern3_pre_reptime` 财报事件,MEA 有 earnings3 数据集)。

In [ ]:
so_layer = mea_prune(so_tracker['next'] + so_tracker['decay'], MEA_PREFIXES, 5)
th_alpha_list = []
for expr, decay in so_layer:
    for alpha in trade_when_factory("trade_when", expr, 'mea'):
        th_alpha_list.append((alpha, decay))
print(len(th_alpha_list))

In [ ]:
# Simulate third order
th_pools = load_task_pool(th_alpha_list, 9, 10)
multi_simulate(th_pools, NEUT, REGION, UNIVERSE, 0)

In [ ]:
# get submitable alphas to check submission
th_tracker = get_alphas(START_D, END_D, 1.58, 1, REGION, 200, "submit")

In [ ]:
stone_bag = []
for alpha in th_tracker['next'] + th_tracker['decay']:
    stone_bag.append(alpha[0])
print(len(stone_bag))
gold_bag = []
check_submission(stone_bag, gold_bag, 0)

In [ ]:
# look date and metrics to locate alphas in the web
view_alphas(gold_bag)